# Lab 6: Fourier Methods, Continuation, and Reduction to the Pole

| | |
|---|---|
| **Module** | M6 — Fourier Methods, Transformations, and Data Enhancement |
| **Estimated time** | ~3 hours |
| **Prerequisites** | M6 lectures (Fourier-domain modeling, transformations, continuation); Homework 6, Problems 6.1–6.2; Lab 3 (total-field anomaly); Lab 5 (inversion) |
| **Textbook** | Blakely Ch. 11, 12 |

---

## Learning Outcomes

By the end of this lab you will be able to:

1. **Apply** the 2D FFT to a gridded potential field and estimate equivalent source depth from the radially averaged power spectrum
2. **Implement** upward continuation in the wavenumber domain and use it for regional–residual separation
3. **Explain** from the filter spectrum why downward continuation is unstable, and **implement** stabilized (cutoff and Wiener) downward continuation
4. **Quantify** noise amplification as a function of continuation distance and signal-to-noise ratio
5. **Design** a reduction-to-pole (RTP) filter and explain why it fails near the magnetic equator

---

## How to use this notebook

- **`[PROVIDED]`** — run as-is
- **`[IMPLEMENT]`** — replace `raise NotImplementedError` with correct code
- **`[VALIDATE]`** — run to check; prints ✓ PASS or ✗ FAIL
- **`[EXPLORE]`** — starting point; modify freely

**Your response:** cells require written answers. Hints: `print(hints['key'])`.


# Lab 6 - Fourier-Domain Processing of Potential Field Data

_You are welcome to use AI tools. All written answers must reference specific numerical results from your notebook — plot features, VALIDATE output, numbers from your calculations. Answers that could have been written without running the notebook will not receive credit for the written component._

## Fourier methods for potential fields

### Background

The Fourier transform is the natural language of potential field processing
because Laplace's equation becomes algebraic in the wavenumber domain.  For
a 2D potential field $F(x,y)$, its Fourier transform is

$$
\tilde{F}(k_x, k_y) = \int\int F(x,y)\,e^{-2\pi i(k_x x + k_y y)}\,dx\,dy
$$

where $k_x$, $k_y$ are spatial frequencies (cycles/m) and
$k = \sqrt{k_x^2 + k_y^2}$ is the radial wavenumber.  Every transformation in
this lab is a multiplication of $\tilde{F}$ by a filter $W(k_x, k_y)$.

The **radially averaged power spectrum** has the property that for a source
ensemble at depth $z_0$, $\ln|\tilde{F}(k)| \propto -2\pi k z_0$ (Blakely §11.4).
Fitting the slope of the log-power spectrum therefore estimates the equivalent
source depth.

### Continuation

From Laplace's equation, the field on a plane a height $h$ above the
observation surface is (Blakely §12.1)

$$
\tilde{F}(k, h) = \tilde{F}(k, 0) \cdot e^{-2\pi k h}
$$

For **upward** continuation ($h > 0$) the factor $e^{-2\pi kh} \leq 1$ and
decays to zero as $k \to \infty$: a stable low-pass filter.  Continuing up to a
height that is large compared to shallow-source depths attenuates the shallow
sources while preserving the deep (regional) signal; the residual
(original minus upward-continued) then highlights the shallow sources.

For **downward** continuation ($h < 0$) the factor becomes
$e^{+2\pi k|h|}$, which **grows exponentially** with $k$.  Any noise present
at high wavenumbers is amplified by this factor — potentially catastrophically.
To make downward continuation tractable we use a **regularized** filter, either a
hard wavenumber cutoff

$$
W_{dc}(k) = \begin{cases}
  e^{+2\pi k|h|} & k \leq k_c \\
  0              & k > k_c
\end{cases}
$$

or a **Wiener-type** stabilizer
$W_{dc} = e^{+2\pi k|h|} / (1 + \varepsilon^2 e^{4\pi k|h|})$,
where $\varepsilon$ is a small noise-to-signal parameter.  Too small a $k_c$
(too large an $\varepsilon$) over-smooths the result; too large a $k_c$ lets
amplified noise dominate.

### Reduction to the pole

**Reduction to the pole (RTP)** converts the total-field magnetic anomaly to what
it would look like if measured at the magnetic pole ($I = 90°$, $D = 0°$), so
anomaly peaks align with source locations.  The RTP filter in the wavenumber
domain is (Blakely §12.3):

$$
W_{RTP}(k_x, k_y) = \frac{1}{\Theta_m\,\Theta_f}
$$

where $\Theta_m$ and $\Theta_f$ are complex factors that depend on the
magnetization direction and field direction.  For induced magnetization,
$\Theta_m = \Theta_f$, and the filter becomes $1/\Theta_f^2$.


In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy.signal import windows

plt.rcParams.update({'figure.dpi': 120, 'axes.grid': False, 'font.size': 11})

def _check(label, got, expected, rtol=1e-4, atol=0.0):
    if np.allclose(got, expected, rtol=rtol, atol=atol):
        print(f'  ✓ PASS  {label}')
    else:
        print(f'  ✗ FAIL  {label}')
        print(f'    expected: {np.asarray(expected)}')
        print(f'    got:      {np.asarray(got)}')

print('Setup complete.')

In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────
hints = {
    'p1_fft2d': (
        "Use np.fft.fft2 for the 2D transform and np.fft.fftshift to center\n"
        "zero-frequency.  Wavenumber arrays:\n"
        "  kx = np.fft.fftfreq(Nx, d=dx)   [cycles/m, unshifted]\n"
        "  ky = np.fft.fftfreq(Ny, d=dy)\n"
        "  KX, KY = np.meshgrid(kx, ky)   -- note Ny rows, Nx cols\n"
        "  K = sqrt(KX^2 + KY^2)          -- radial wavenumber\n"
        "Apply fftshift to both the spectrum and K before plotting."
    ),
    'p1_spectrum_depth': (
        "Radially average the log power spectrum: for each annulus of K,\n"
        "average log|F_tilde|^2 over all directions.\n"
        "Then fit log(P) vs K with a straight line: slope = -4*pi*z_eq\n"
        "(the 4*pi comes from 2*pi*K = angular wavenumber; check Blakely convention).\n"
        "Use np.polyfit on the linear portion of the spectrum."
    ),
    'p2_upward': (
        "Upward continuation by height h (meters):\n"
        "  1. F_tilde, KX, KY, K = compute_fft2(data, dx, dy)\n"
        "  2. Filter: F_tilde_cont = F_tilde * continuation_filter(K, h)\n"
        "  3. Result: np.fft.ifft2(F_tilde_cont).real\n"
        "Note: K is NOT shifted when multiplying — use fftfreq (unshifted)."
    ),
    'p2_regional': (
        "Strategy: upward continue the total field to height h_reg >> depth_shallow.\n"
        "The upward-continued field approximates the regional (deep source only).\n"
        "The residual = original - upward_continued is dominated by shallow sources.\n"
        "Choose h_reg empirically: too small = shallow source bleeds into regional;\n"
        "too large = deep source is also attenuated."
    ),
    'p3_filter_plot': (
        "Plot exp(-2*pi*k*h) for h = 500, 1000, 2000 m (upward)\n"
        "and exp(+2*pi*k*h) for the same h (downward, no cutoff).\n"
        "k ranges from 0 to k_Nyquist = 1/(2*dx_prof).\n"
        "Use a log scale on y to make the exponential growth visible."
    ),
    'p3_cutoff_choice': (
        "The amplification factor at cutoff k_c is exp(2*pi*k_c*h).\n"
        "For the result to be useful, you want this factor to be at most\n"
        "the signal-to-noise ratio (SNR).\n"
        "If SNR = 100 and h = 1 km:\n"
        "  k_c = ln(SNR) / (2*pi*h) = ln(100) / (2*pi*1000) ≈ 7.3e-4 cycles/m\n"
        "i.e., k_c ≈ 1 cycle per 1370 m."
    ),
    'p3_wiener': (
        "Wiener stabilizer:\n"
        "  W(k) = exp(+2*pi*k*h) / (1 + eps^2 * exp(4*pi*k*h))\n"
        "where eps is the noise-to-signal amplitude ratio (e.g., 0.01 for SNR=100).\n"
        "As k -> 0: W -> exp(2*pi*k*h) * 1 (exact downward continuation)\n"
        "As k -> inf: W -> 1/eps^2 * exp(-2*pi*k*h) (rolls off)"
    ),
    'p4_rtp_filter': (
        "For induced magnetization (magnetization = field direction) and 2D case:\n"
        "  I = inclination, D = declination (both in radians)\n"
        "  Theta_f = sin(I) + i*cos(I)*(kx*cos(D) + ky*sin(D)) / K\n"
        "         [see Blakely eq. 12.20; be careful with D convention]\n"
        "  W_RTP = 1 / Theta_f^2  (for induced mag)\n"
        "  Set W_RTP = 0 where K = 0 (DC term) to avoid division by zero.\n"
        "  The filter blows up when Theta_f -> 0, i.e., near the equator."
    ),
    'p4_rtp_instability': (
        "Near the magnetic equator (I small), the denominator Theta_f -> 0\n"
        "for wavenumbers perpendicular to the field (kx direction).\n"
        "This means the RTP filter amplifies short-wavelength noise in that\n"
        "direction -> 'ringing' artifacts oriented N-S."
    ),
}
# print(hints['p2_upward'])

---
## Part 1: The 2D Amplitude Spectrum *(Guided — ~35 min)*

Before filtering we need to understand the data in the frequency domain.
We compute the 2D amplitude spectrum of a synthetic magnetic grid, examine its
structure, and estimate a characteristic source depth from the power spectrum slope.
The same grid is used again in Parts 2 and 4.

**Available hints:** `p1_fft2d`, `p1_spectrum_depth`


In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────
# Synthetic 2D magnetic anomaly grid.
#
# Three buried dipolar bodies at different depths and locations.
# Regional field: I = 65°, D = 0°.

mu0 = 4 * np.pi * 1e-7

# Grid
dx = dy = 500.0   # m, grid spacing
Nx = Ny = 128
x1d = np.arange(Nx) * dx
y1d = np.arange(Ny) * dy
X, Y = np.meshgrid(x1d, y1d)   # shape (Ny, Nx)
x0 = x1d.mean()
y0 = y1d.mean()

I_reg = np.radians(65.0)   # regional inclination
D_reg = 0.0                 # regional declination

def _dipole_dF_2d(X, Y, xc, yc, depth, m_moment, I_field):
    """Total-field anomaly (nT) from a buried dipole in 3D geometry."""
    Ix = np.cos(I_field)   # northward (x-direction)
    Iz = -np.sin(I_field)  # upward  (z-up convention)
    rx = X - xc
    ry = Y - yc
    rz = depth   # upward distance = depth (source is below)
    r2 = rx**2 + ry**2 + rz**2
    r  = np.sqrt(r2)
    mr = Ix*rx + Iz*rz
    pref = (mu0/(4*np.pi)) * m_moment / r**5
    dBx = pref*(3*mr*rx - Ix*r2)
    dBz = pref*(3*mr*rz - Iz*r2)
    dF = dBx*np.cos(I_field) + dBz*(-np.sin(I_field))
    return dF * 1e9

# Three sources
rng = np.random.default_rng(55)
F_grid  = _dipole_dF_2d(X, Y, x0 - 8e3, y0 + 5e3, 1500.0, 3e12, I_reg)
F_grid += _dipole_dF_2d(X, Y, x0 + 6e3, y0 - 4e3, 4000.0, 8e12, I_reg)
F_grid += _dipole_dF_2d(X, Y, x0 + 2e3, y0 + 8e3, 2500.0, 5e12, I_reg)
F_grid += rng.normal(0, 1.0, size=F_grid.shape)   # 1 nT noise

print(f'Grid: {Nx} × {Ny}, spacing {dx} m')
print(f'Anomaly range: {F_grid.min():.1f} to {F_grid.max():.1f} nT')

In [ ]:
# [IMPLEMENT] ─────────────────────────────────────────────────────────────────
# Tier 1: Compute the 2D Fourier transform and wavenumber arrays.

def compute_fft2(data, dx, dy):
    """
    Compute 2D FFT and associated wavenumber grids.

    Parameters
    ----------
    data : ndarray, shape (Ny, Nx)
        Gridded field values.
    dx, dy : float
        Grid spacing in x and y directions, meters.

    Returns
    -------
    F_tilde : ndarray, shape (Ny, Nx), complex
        2D Fourier transform of data (NOT shifted).
    KX : ndarray, shape (Ny, Nx)
        Wavenumber in x, cycles/m (NOT shifted).
    KY : ndarray, shape (Ny, Nx)
        Wavenumber in y, cycles/m (NOT shifted).
    K : ndarray, shape (Ny, Nx)
        Radial wavenumber sqrt(KX^2+KY^2), cycles/m (NOT shifted).

    Notes
    -----
    Use np.fft.fft2 and np.fft.fftfreq.
    KX varies along axis=1 (columns); KY varies along axis=0 (rows).
    Return the UNSHIFTED arrays — the filter functions use these directly.
    """
    Ny, Nx = data.shape

    # Step 1: 2D FFT (unshifted)
    # YOUR CODE HERE
    raise NotImplementedError

    # Step 2: wavenumber arrays
    # YOUR CODE HERE
    raise NotImplementedError

    # Step 3: radial wavenumber
    # YOUR CODE HERE
    raise NotImplementedError

    return F_tilde, KX, KY, K

In [ ]:
# [VALIDATE] ──────────────────────────────────────────────────────────────────

F_tilde, KX, KY, K = compute_fft2(F_grid, dx, dy)

# Test 1: Parseval's theorem — energy in space vs. frequency domain
_E_space = np.sum(F_grid**2)
_E_freq  = np.sum(np.abs(F_tilde)**2) / (Nx * Ny)   # normalize
_check('Parseval: E_space ≈ E_freq/N', _E_space, _E_freq, rtol=1e-6)

# Test 2: DC component equals sum of data
_check('DC term = sum(data)', np.abs(F_tilde[0, 0]), abs(F_grid.sum()), rtol=1e-8)

# Test 3: wavenumber shape
_check('KX shape', KX.shape, (Ny, Nx), atol=0)

# Test 4: Nyquist frequency
_kx_nyquist_expected = 1.0 / (2.0 * dx)
_check('Nyquist kx', np.abs(KX).max(), _kx_nyquist_expected, rtol=1e-10)

In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Make a 2-panel figure:
#   Left panel: The F_grid anomaly map (nT). Label axes in km.
#   Right panel: The 2D log10 amplitude spectrum (shifted to center DC).
#                Label axes in cycles/km. Use a perceptually uniform colormap.
#
# Add a colorbar to each panel.

# For the spectrum: np.fft.fftshift(np.log10(np.abs(F_tilde) + 1))

# YOUR CODE HERE

In [ ]:
# [IMPLEMENT] ─────────────────────────────────────────────────────────────────
# Tier 1: Radially averaged power spectrum and depth estimate.

def radial_power_spectrum(F_tilde, K, dk=None):
    """
    Compute the radially averaged log power spectrum.

    Parameters
    ----------
    F_tilde : ndarray, complex
        2D Fourier transform (unshifted).
    K : ndarray
        Radial wavenumber, cycles/m (unshifted, same shape as F_tilde).
    dk : float, optional
        Bin width in cycles/m.  Default: minimum non-zero K.

    Returns
    -------
    k_centers : ndarray
        Center wavenumber of each bin, cycles/m.
    log_power : ndarray
        Mean log10 power in each bin: log10(mean(|F_tilde|^2)).

    Notes
    -----
    Bin K into annuli of width dk.  Skip the DC bin (K=0).
    Return k_centers and log_power only for bins with at least 4 entries.
    """
    if dk is None:
        dk = K[K > 0].min()

    # Step 1: compute power |F_tilde|^2
    # YOUR CODE HERE
    raise NotImplementedError

    # Step 2: bin by K and average log power
    # YOUR CODE HERE
    raise NotImplementedError

    return k_centers, log_power

In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Compute the radial power spectrum and estimate source depth.
#
# 1. Call radial_power_spectrum(F_tilde, K)
# 2. Plot log_power vs k_centers (in cycles/km)
# 3. Fit a straight line to the low-wavenumber portion (k < 0.5 cycles/km)
#    Relationship: log10(P) ≈ const - (4*pi*log10(e)) * z_eq * k
#    => z_eq = -slope / (4*pi*log10(e))
# 4. Overplot the fit and annotate with the estimated depth
# 5. Compare to the range of true source depths (1.5, 2.5, 4.0 km)

# YOUR CODE HERE

### Question 1.1 — Interpreting the spectrum

The power spectrum has a steep roll-off at high wavenumbers.  Is this entirely
due to the depths of the sources, or could other factors contribute?
The spectral depth estimate gives a single equivalent depth — where does it
fall relative to the three true depths (1.5, 2.5, 4.0 km)?  Is this expected?

**Your response:**

> *(Write 4–5 sentences.)*


---
## Part 2: Upward Continuation and Regional–Residual Separation *(Supported — ~40 min)*

Upward continuation smooths the field and suppresses short-wavelength
noise and shallow sources — the standard tool for separating regional and
residual anomalies.  We first write a general continuation filter (which we
will reuse for downward continuation in Part 3), then apply it to the 2D grid.

**Available hints:** `p2_upward`, `p2_regional`


In [ ]:
# [IMPLEMENT] ─────────────────────────────────────────────────────────────────
# Tier 1: Continuation filter amplitude (used for both upward and downward).

def continuation_filter(k, h):
    """
    Continuation filter amplitude in the wavenumber domain.

    Parameters
    ----------
    k : float or ndarray
        Radial wavenumber, cycles/m.  Must be >= 0.
    h : float
        Continuation height, meters.
        Positive = upward (stable, < 1).
        Negative = downward (unstable, > 1 for |k| > 0).

    Returns
    -------
    W : float or ndarray
        Filter amplitude.  W = exp(-2*pi*k*h).

    Notes
    -----
    - For h > 0: W < 1 for all k > 0 (stable low-pass)
    - For h < 0: W > 1 for all k > 0 (unstable amplification)
    - At k = 0 (DC): W = 1 always
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# [VALIDATE] ──────────────────────────────────────────────────────────────────

# At k=0: filter = 1 regardless of h
_check('W(k=0, h=1000) = 1', continuation_filter(0.0, 1000.0), 1.0)
_check('W(k=0, h=-500) = 1', continuation_filter(0.0, -500.0), 1.0)

# Upward h=500: at k=1/1000 (wavelength 1 km), W = exp(-pi) ≈ 0.0432
_k_test = 1.0/1000.0   # cycles/m
_W_up = continuation_filter(_k_test, 500.0)
_check('Upward W at k=1/1000, h=500', _W_up, np.exp(-2*np.pi*_k_test*500.0), rtol=1e-10)

# Downward: W_down = 1/W_up for same |h|
_W_down = continuation_filter(_k_test, -500.0)
_check('Downward W = 1/upward W', _W_down, 1.0/_W_up, rtol=1e-10)

# Upward W should be < 1 for k > 0, h > 0
_k_arr = np.linspace(1e-5, 2e-3, 100)
if np.all(continuation_filter(_k_arr, 1000.0) < 1.0):
    print('  ✓ PASS  Upward filter < 1 for all k > 0')
else:
    print('  ✗ FAIL  Upward filter should be < 1 for k > 0')

In [ ]:
# [IMPLEMENT] ─────────────────────────────────────────────────────────────────
# Tier 2: 2D upward continuation.

def upward_continuation(data, dx, dy, h):
    """
    Upward continue a gridded potential field by height h.

    Parameters
    ----------
    data : ndarray, shape (Ny, Nx)
        Gridded field values.
    dx, dy : float
        Grid spacing, meters.
    h : float
        Continuation height, meters (positive = upward).

    Returns
    -------
    data_cont : ndarray, shape (Ny, Nx)
        Upward-continued field.

    Notes
    -----
    Use compute_fft2 (Part 1), multiply by continuation_filter(K, h),
    and inverse transform.  K must be the UNSHIFTED radial wavenumber (cycles/m).
    """
    raise NotImplementedError

In [ ]:
# [VALIDATE] ──────────────────────────────────────────────────────────────────

# Upward continuation must not increase total power
_F_cont_1 = upward_continuation(F_grid, dx, dy, 500.0)
_F_cont_5 = upward_continuation(F_grid, dx, dy, 5000.0)

if np.std(_F_cont_1) < np.std(F_grid):
    print('  ✓ PASS  Continuation reduces amplitude (h=500 m)')
else:
    print('  ✗ FAIL  Continuation should reduce amplitude')

if np.std(_F_cont_5) < np.std(_F_cont_1):
    print('  ✓ PASS  Higher continuation reduces more (h=5000 > h=500)')
else:
    print('  ✗ FAIL  Higher continuation should reduce more')

# DC value (mean) should be unchanged by upward continuation
_check('Mean preserved by continuation', _F_cont_5.mean(), F_grid.mean(), rtol=1e-5)

# Upward continuing by h=0 should return the original
_F_cont_0 = upward_continuation(F_grid, dx, dy, 0.0)
_check('h=0 returns original', _F_cont_0, F_grid, rtol=1e-6)

In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Apply upward continuation at three heights: 500 m, 2000 m, 5000 m.
#
# Make a 2×2 figure:
#   [0,0] Original field
#   [0,1] h = 500 m
#   [1,0] h = 2000 m
#   [1,1] h = 5000 m
#
# Use the same colorscale limits for all panels (vmin/vmax from original).
# Title each panel with the height and the RMS amplitude (np.std).
#
# Also compute the RESIDUAL (original minus h=2000 m continuation)
# and show it in a 5th panel. What does the residual represent geologically?

# YOUR CODE HERE

### Question 2.1 — Geological interpretation

The residual anomaly (original minus upward-continued) isolates which
sources from the three in the synthetic dataset?  Which sources survive
at 5 km continuation?  Explain in terms of the continuation filter and
source depth.

**Your response:**

> *(Write 4–6 sentences.  Reference specific source depths from the dataset.)*


### Question 2.2 — Limits of the method

The upward-continuation approach to regional-residual separation has a
fundamental limitation: it cannot separate two sources at the same depth.
Give a concrete example of a geological scenario where you would face this
problem, and describe an alternative strategy that might work in that case.

**Your response:**

> *(Write 4–5 sentences.)*


---
## Part 3: Downward Continuation and Stability *(Supported — ~55 min)*

Downward continuation moves the observation plane closer to the sources,
sharpening anomalies — but it reverses the low-pass filter you used in Part 2,
so it amplifies noise.  To see exactly what is gained and lost, we switch to a
1D gravity profile where the true field at depth is known from the forward model.

The sources here are **2D horizontal cylinders** (infinitely long
perpendicular to the profile).  This matters: the 1D continuation filter
$e^{-2\pi |k_x| h}$ is exact only for 2D sources.  A 3D body such as a sphere
also has wavenumber content in $k_y$, so continuing a single profile across it
would not reproduce its true field at depth.

**Available hints:** `p3_filter_plot`, `p3_cutoff_choice`, `p3_wiener`


In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────
# Synthetic 1D gravity profile across two overlapping 2D sources
# (horizontal cylinders, infinitely long perpendicular to the profile):
#   - Shallow source: cylinder at depth 1.5 km, radius 500 m, drho = 300 kg/m^3
#   - Deep source (regional): cylinder at depth 10 km, radius 4000 m, drho = 50 kg/m^3

G_grav  = 6.674e-11
dx_prof = 200.0                             # m, profile spacing
x_prof  = np.linspace(-20e3, 20e3, 201)     # m

def _cylinder_gz_1d(x, depth, R, drho):
    lam = np.pi * R**2 * drho               # mass per unit length, kg/m
    return 2 * G_grav * lam * depth / (x**2 + depth**2)

z_shallow, R_shallow, drho_shallow = 1500.0, 500.0, 300.0

rng_prof   = np.random.default_rng(99)
g_shallow  = _cylinder_gz_1d(x_prof, z_shallow, R_shallow, drho_shallow) * 1e5   # mGal
g_deep     = _cylinder_gz_1d(x_prof, 10000.0, 4000.0, 50.0) * 1e5               # mGal
noise_sigma = 0.02   # mGal
g_total    = g_shallow + g_deep + rng_prof.normal(0, noise_sigma, size=x_prof.shape)

print(f'Shallow source peak: {g_shallow.max():.4f} mGal')
print(f'Deep source peak:    {g_deep.max():.4f} mGal')
print(f'Total range:         {g_total.min():.4f} to {g_total.max():.4f} mGal')

# Wavenumber array for the profile
k_prof = np.fft.fftfreq(len(x_prof), d=dx_prof)   # cycles/m

In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Plot the continuation filter amplitudes (use continuation_filter) to visualize stability.
#
# Make two panels:
#
# Panel 1 (linear scale): Upward continuation filters W(k) for h = 500, 1000, 2000 m.
#   x-axis: wavenumber in cycles/km,  y-axis: filter amplitude (linear)
#   Shade the area k > k_Nyquist in gray.
#
# Panel 2 (log y-scale): Downward continuation filter exp(+2*pi*k*h) for same |h|.
#   Also show the SNR-based cutoff for SNR=100 and h=1000 m:
#     k_c = ln(100) / (2*pi*1000) [cycles/m]
#   Mark k_c with a vertical dashed line.

k_plot = np.linspace(0, 1/(2*dx_prof), 500)   # cycles/m up to Nyquist

# YOUR CODE HERE

### Question 3.1 — Quantifying instability

From your filter plot: at $h = 1000$ m and $k = k_{Nyquist}$ (the shortest
resolvable wavelength at $\Delta x = 200$ m), what is the amplification factor
of the downward filter?

If the data noise has amplitude $\sigma = 0.02$ mGal and the signal amplitude
is $\sim 2$ mGal, at what wavenumber $k_c$ does the downward-continued noise
equal the signal amplitude?  Show your calculation, and show that it reduces to
$k_c = \ln(\text{SNR}) / (2\pi h)$ — the cutoff you will use below.  What
wavelength does this $k_c$ correspond to, and what is the main drawback of
choosing the cutoff this way?

**Your response:**

> *(Write 4–6 sentences, including the calculation.)*


In [ ]:
# [IMPLEMENT] ─────────────────────────────────────────────────────────────────
# Tier 2: Stabilized downward continuation (1D).

def downward_continuation_cutoff(data, dx, h_down, k_cutoff):
    """
    Downward continue a 1D field using a wavenumber cutoff.

    Parameters
    ----------
    data : ndarray, shape (N,)
        Field values along profile.
    dx : float
        Spatial sampling interval, meters.
    h_down : float
        Downward distance (positive = downward = deeper), meters.
    k_cutoff : float
        Wavenumber cutoff, cycles/m.  Filter = 0 for |k| > k_cutoff.

    Returns
    -------
    data_down : ndarray
        Downward-continued field.

    Notes
    -----
    Filter: W(k) = exp(+2*pi*|k|*h_down) for |k| <= k_cutoff, else 0.
    h_down > 0 means moving the observation level downward (toward sources).
    """
    raise NotImplementedError


def downward_continuation_wiener(data, dx, h_down, eps):
    """
    Downward continue using a Wiener-type stabilizer.

    Parameters
    ----------
    data : ndarray, shape (N,)
    dx : float
        Sampling interval, meters.
    h_down : float
        Downward distance, meters (positive = deeper).
    eps : float
        Noise-to-signal amplitude ratio (e.g., 0.01 for SNR=100).

    Returns
    -------
    data_down : ndarray

    Notes
    -----
    W(k) = exp(+2*pi*|k|*h_down) / (1 + eps^2 * exp(4*pi*|k|*h_down))
    At k=0: W=1.  As k->inf: W->0 (stabilized).
    """
    raise NotImplementedError

In [ ]:
# [VALIDATE] ──────────────────────────────────────────────────────────────────

# Upward-then-downward should approximately recover the original (within the cutoff bandwidth)
from numpy.fft import fft, ifft, fftfreq

def _upward_1d(data, dx, h):
    k = fftfreq(len(data), d=dx)
    W = np.exp(-2*np.pi*np.abs(k)*h)
    return ifft(fft(data)*W).real

_h_test   = 1000.0
_k_cutoff = np.log(50.0) / (2*np.pi*_h_test)   # SNR=50

# Apply upward continuation then downward continuation
_g_up   = _upward_1d(g_total, dx_prof, _h_test)
_g_down_co = downward_continuation_cutoff(_g_up, dx_prof, _h_test, _k_cutoff)
_g_down_wi = downward_continuation_wiener(_g_up, dx_prof, _h_test, eps=0.02)

# The low-wavenumber content should be recovered (check variance)
_var_orig  = np.var(g_total)
_var_round = np.var(_g_down_co)
if 0.1*_var_orig < _var_round < 2.0*_var_orig:
    print(f'  ✓ PASS  Round-trip variance reasonable: {_var_round:.4f} vs {_var_orig:.4f}')
else:
    print(f'  ✗ FAIL  Round-trip variance out of range: {_var_round:.4f} vs {_var_orig:.4f}')

# Wiener filter at k=0: should equal the DC value
_dc_orig = g_total.mean()
_dc_wi   = _g_down_wi.mean()
_check('Wiener: DC preserved', _dc_wi, _dc_orig, rtol=0.01)

# Downward continuation without cutoff should blow up
_g_no_cutoff = downward_continuation_cutoff(g_total, dx_prof, 5000.0, k_cutoff=1/(2*dx_prof))
if np.max(np.abs(_g_no_cutoff)) > 1e6:
    print('  ✓ PASS  Uncutoff downward continuation explodes (as expected)')
else:
    print('  ~ NOTE  Uncutoff continuation did not explode — check if you applied k_cutoff=Nyquist')

In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Noise amplification study: how well can we re-image the shallow source
# from noisy surface data, and how should the cutoff be chosen?
#
# 1. Start with g_shallow (the clean, noise-free shallow-source profile).
# 2. For each sigma in noise_levels, add Gaussian noise to g_shallow.
# 3. Downward continue each noisy profile by h_down_target = 750 m.
#    (The top of the cylinder is at 1000 m depth, so this plane stays above
#    the source — continuing into or through a source is not valid.)
#    Use the SNR-based cutoff  k_c = ln(SNR) / (2*pi*h_down_target)
#    where SNR = peak(g_shallow) / sigma.
# 4. Repeat step 3 with the Wiener stabilizer, using eps = 1/SNR.
# 5. Compute the RMS error of each result against g_true_at_depth (the exact
#    field at 750 m depth from the forward model).  As a baseline, also compute
#    the RMS error of doing nothing: g_shallow vs. g_true_at_depth.
# 6. For sigma = 0.02 mGal, scan the cutoff k_c = f * ln(SNR)/(2*pi*h) for
#    f between 0.2 and 1.0, and find the f that minimizes the RMS error.
#
# Requirements:
#   - Show the 4 cutoff-based profiles (step 3) in a single figure, with
#     g_true_at_depth as a dashed black reference
#   - Plot RMS error vs. noise sigma for both stabilizers on one set of axes,
#     with the "no continuation" baseline as a horizontal line
#   - Plot RMS error vs. f for the step-6 scan and overlay the best profile
#     on g_true_at_depth

h_down_target = 750.0                         # m
noise_levels  = [0.005, 0.01, 0.02, 0.05]     # mGal
g_true_at_depth = _cylinder_gz_1d(x_prof, z_shallow - h_down_target,
                                  R_shallow, drho_shallow) * 1e5   # mGal

# YOUR CODE HERE

### Question 3.2 — Choosing the cutoff

How did the RMS error of the SNR-based cutoff compare with the "no
continuation" baseline?  What value of $f$ minimized the error in your scan, and
what RMS error did it reach?  Explain why $k_c = \ln(\text{SNR})/(2\pi h)$ is too
aggressive: what does it imply about the size of the amplified noise just
below $k_c$, and how many Fourier components contribute noise of that size?

**Your response:**

> *(Write 4–6 sentences. Include RMS error numbers from your calculations.)*


### Question 3.3 — Wiener vs. cutoff

Compare the cutoff-based and Wiener-based downward continuation results.
Which produces fewer artifacts, and which has the lower RMS error at each
noise level?  How do the two filters treat wavenumbers near $k_c$, and how
does that explain any ringing you see?  Why might the Wiener stabilizer be
preferred in practice despite being slightly more complex to implement?

**Your response:**

> *(Write 3–5 sentences. Include RMS error numbers from your calculations.)*


---
## Part 4: Reduction to the Pole *(Open — ~40 min)*

The total-field anomaly has a characteristic asymmetry that depends on
the regional field inclination (as you showed in Lab 3).  Reduction to
the pole (RTP) removes this asymmetry so that anomaly peaks locate
directly above sources.

**Available hints:** `p4_rtp_filter`, `p4_rtp_instability`


### Task 4.1 — Implement and apply the RTP filter

Implement the 2D RTP filter for induced magnetization.  Apply it to `F_grid`
(which was generated at $I = 65°$, $D = 0°$).  Verify that anomaly peaks in
the RTP map align with the known source locations.

Then repeat the RTP for $I = 10°$ (near-equatorial).  Show the result and
explain any artifacts.


In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Implement the RTP filter and plot results.
#
# True source locations (for comparison):
source_xy = [
    (x0 - 8e3, y0 + 5e3),   # source 1, depth 1.5 km
    (x0 + 6e3, y0 - 4e3),   # source 2, depth 4.0 km
    (x0 + 2e3, y0 + 8e3),   # source 3, depth 2.5 km
]

# Steps:
# 1. Compute KX, KY, K from compute_fft2
# 2. Compute Theta_f for I=65°, D=0°
# 3. Build RTP filter W_RTP = 1 / Theta_f^2
# 4. Set W_RTP[K==0] = 0
# 5. Apply: F_rtp = ifft2(fft2(F_grid) * W_RTP).real
# 6. Plot F_grid and F_rtp side by side; mark source locations
# 7. Repeat for I=10°

# YOUR CODE HERE

### Question 4.1 — RTP at low inclination

Describe the artifacts that appear in the $I = 10°$ RTP result.
What aspect of the filter formula causes them?  Geophysically, what
would you do if you needed to apply RTP to data collected near the
magnetic equator?

**Your response:**

> *(Write 4–6 sentences.)*


### Question 4.2 — Remanent magnetization

The RTP filter above assumes that magnetization is aligned with the ambient
field (induced magnetization).  If the body has remanent magnetization
in a different direction (as is common in oceanic crust), how would the
RTP result be different?  Could you detect the presence of remanence
from the shape of the anomaly alone?

**Your response:**

> *(Write 4–5 sentences.)*


---
## Synthesis

### S1 — Filter interpretation

You have now used several wavenumber-domain filters: upward continuation
($e^{-2\pi kh}$), stabilized downward continuation ($e^{+2\pi kh}$ with a cutoff
or Wiener taper), RTP ($1/\Theta_f^2$), and implicitly the derivative operators
($i2\pi k_x$ for the horizontal derivative).  Explain the key difference between
an **amplitude** filter (like continuation) and a **phase** filter (like RTP)
in terms of what they do to the amplitude and phase of the spectrum.  Give one
practical advantage and one practical risk for each type, using results from
Parts 2–4.

**Your response:**

> *(Write 4–6 sentences.)*

### S2 — Practical recommendations

A colleague asks you: "I have ground-based gravity data with station spacing
100 m and noise $\sigma = 0.05$ mGal.  I want to downward continue to 500 m
depth to better image a shallow target.  Is this feasible?"

Using the cutoff-wavenumber formula, estimate the spatial resolution
(minimum resolvable wavelength) of the downward-continued result.
Is this useful resolution for a target at 500 m depth?  What would you
recommend?

**Your response:**

> *(Write 4–6 sentences. Include a calculation.)*

### S3 — Continuation vs. inversion

Both downward continuation (Part 3) and inversion (Lab 5) attempt to recover
subsurface structure from surface measurements, and both need regularization.
Compare the two approaches: what does each assume about the source geometry,
what type of prior information does each require, and when would you choose
one over the other?

**Your response:**

> *(Write 5–7 sentences.)*


---
## Extensions *(optional)*

### E1 — Analytic signal

The analytic signal amplitude $|A| = \sqrt{(\partial F/\partial x)^2 + (\partial F/\partial y)^2 + (\partial F/\partial z)^2}$
peaks over source edges regardless of magnetization direction.  Implement it
using the Fourier-domain vertical derivative ($2\pi k\,\tilde{F}$) and the
horizontal derivatives, and compare peak locations to the source positions.

### E2 — Tilt derivative

The tilt angle $\tau = \arctan(\partial F/\partial z / |\nabla_H F|)$ is
approximately zero over source edges and peaks over source centers.  Implement
it and compare it to the RTP map.  Which gives a sharper localization of sources?

### E3 — Edge tapering

Real survey grids have finite extent, causing edge effects in the FFT.
Apply a cosine taper (20% of the grid width, e.g. `windows.tukey`) to `F_grid`
before computing the FFT.  How does tapering affect the upward continuation
and RTP results?

### E4 — Iterative downward continuation

An alternative to the direct inverse filter is iterative upward continuation:
start with the surface data as the initial guess for the depth-level field,
then iteratively correct using the difference between the upward-continued
trial field and the observed data.  Implement 5–10 iterations on the Part 3
profile and compare the result to the direct Wiener filter.


In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Extension workspace.

# YOUR CODE HERE